# Assemble .fcs files and metadata from the PT1 panel

## Imports and output

In [1]:
from datetime import date # date for uploads

import hisepy # HISE SDK

import polars as pl # data frame handling
import flowio # flow cytometry file handling

import re # regular expressions

import os # file/path utilities
import shutil # file utilities
import tarfile # file bundling in .tar files

In [2]:
if not os.path.isdir('output'):
    os.mkdir('output')

## Helper functions

Generate a unique ID for file deposit

In [3]:
def element_id(n = 3):
    import periodictable
    from random import randrange
    rand_el = []
    for i in range(n):
        el = randrange(0,118)
        rand_el.append(periodictable.elements[el].name)
    rand_str = '-'.join(rand_el)
    return rand_str

Helpers for finding the most recent batch and file timestamp to ensure we have the most recent version of our files

In [4]:
def get_latest_batch(df):
    batches = list(df['file.batchID'].unique())
    batches.sort()
    return batches[-1]

In [5]:
def file_to_time(fn):
    file_time = re.sub('AIFI-(.+)Z/.+','\\1',fn)
    file_time = re.sub('UCSD-(.+)Z/.+','\\1',file_time)
    file_time = re.sub('\\..+','',file_time)
    return file_time

Helpers for file caching. By default, `hisepy.cache_files` will re-retrieve each file, even if we've cached it before.

We'll also do our file caching queries in chunks of 50 files.

In [6]:
def find_nested_file(path):
    if os.path.isdir(path):
        subpath = os.listdir(path)[0]
        path = f'{path}/{subpath}'
        return find_nested_file(path)
    else:
        return path

In [7]:
def cache_chunk_once(ids):    
    # get cached files
    in_root = '/home/workspace/input/1918706177/'
    projects = os.listdir(in_root)
    cached_ids = {}
    for project in projects:
        project_ids = os.listdir(f'{in_root}/{project}')
        for project_id in project_ids:
            cached_ids[project_id] = f'{in_root}/{project}/{project_id}'
    
    cached_paths = []
    not_cached = []
    for uuid in ids:
        if not uuid in cached_ids.keys():
            not_cached.append(uuid)
        else:
            cache_path = find_nested_file(cached_ids[uuid])
            cached_paths.append(cache_path)

    if len(not_cached) > 0:
        new_cache = hisepy.cache_files(not_cached)
        cached_paths = cached_paths + new_cache
    
    return cached_paths

## Read sample metadata

A few adjustments need to be made to use the metadata:
- Trim drawDate to drawYear
- Add subject.ageGroup
- Select relevant columns

In [8]:
meta_uuid = 'af25e3e7-25c1-4476-afb4-926bd201db8f'

In [9]:
meta_file = hisepy.cache_files([meta_uuid])[0]

2026-03-14 14:09:07,395 INFO [hisepy.logging:175] logging 3852 133323142518592 Calling cache_files
2026-03-14 14:09:11,427 INFO [hisepy.logging:208] logging 3852 133323142518592 Finished cache_files, success=True, time_elapsed=2.508s


In [10]:
meta = pl.read_csv(meta_file)

In [11]:
meta.shape

(868, 19)

### drawDate -> drawYear

In [12]:
meta = meta.with_columns(
    pl.col('sample.drawDate').str.replace('-.+','')
).rename({'sample.drawDate':'sample.drawYear'})

### Add age groups

In [13]:
age_groups = {
    'BR1': 'Young Adult',
    'BR2': 'Older Adult'
}

In [14]:
meta = meta.with_columns(
    pl.Series(
        name = 'subject.ageGroup',
        values = [age_groups[c] for c in meta['cohort.cohortGuid']]
    )
)

### Select columns

In [15]:
keep_meta_cols = [
    'cohort.cohortGuid',
    'subject.subjectGuid',
    'sample.sampleKitGuid',
    'subject.biologicalSex',
    'subject.birthYear',
    'subject.ageAtFirstDraw',
    'subject.ageGroup',
    'subject.race',
    'subject.ethnicity',
    'subject.cmv',
    'sample.visitName',
    'sample.drawYear',
    'sample.subjectAgeAtDraw',
    'sample.daysSinceFirstVisit'
]

In [16]:
meta = meta.select(keep_meta_cols)

In [17]:
meta.head()

cohort.cohortGuid,subject.subjectGuid,sample.sampleKitGuid,subject.biologicalSex,subject.birthYear,subject.ageAtFirstDraw,subject.ageGroup,subject.race,subject.ethnicity,subject.cmv,sample.visitName,sample.drawYear,sample.subjectAgeAtDraw,sample.daysSinceFirstVisit
str,str,str,str,i64,i64,str,str,str,str,str,str,i64,i64
"""BR1""","""BR1001""","""KT00001""","""Female""",1987,32,"""Young Adult""","""Caucasian""","""Non-Hispanic origin""","""Negative""","""Flu Year 1 Day 0""","""2019""",32,0
"""BR1""","""BR1002""","""KT00002""","""Male""",1991,28,"""Young Adult""","""Caucasian""","""Non-Hispanic origin""","""Negative""","""Flu Year 1 Day 0""","""2019""",28,0
"""BR1""","""BR1003""","""KT00003""","""Female""",1989,30,"""Young Adult""","""Caucasian""","""Non-Hispanic origin""","""Negative""","""Flu Year 1 Day 0""","""2019""",30,0
"""BR1""","""BR1004""","""KT00004""","""Male""",1989,30,"""Young Adult""","""Caucasian""","""Non-Hispanic origin""","""Negative""","""Flu Year 1 Day 0""","""2019""",30,0
"""BR1""","""BR1005""","""KT00006""","""Female""",1992,27,"""Young Adult""","""Caucasian""","""Non-Hispanic origin""","""Negative""","""Flu Year 1 Day 0""","""2019""",27,0


## Get panel definition
We'll include the panel composition in the output files for convenience

In [18]:
panel_uuid = "d4349731-4eae-4381-8519-72816b0cb9e7"
panel_csv = hisepy.cache_files([panel_uuid])[0]
panel_meta = pl.read_csv(panel_csv)

2026-03-14 14:09:11,583 INFO [hisepy.logging:175] logging 3852 133323142518592 Calling cache_files
2026-03-14 14:09:18,997 INFO [hisepy.logging:208] logging 3852 133323142518592 Finished cache_files, success=True, time_elapsed=2.421s


## Locate FCS files in HISE

For each sample, we'll need to filter for the file from the latest batch and with the most recent timestamp.

We'll work through the sample kit IDs in chunks to help manage our queries to HISE for so many files.

In [19]:
meta_chunks = meta.iter_slices(n_rows = 50)

In [20]:
pt1_list = []
for chunk in meta_chunks:
    chunk_files = hisepy.get_file_descriptors(
        query_dict = {
            'fileType': ['FlowCytometry'],
            'sampleKitGuid': chunk['sample.sampleKitGuid'].to_list(),
            'panel': ['PT1']
        }
    )['descriptors']
    pt1_list.append(pl.DataFrame(chunk_files))

2026-03-14 14:09:19,050 INFO [hisepy.logging:175] logging 3852 133323142518592 Calling get_file_descriptors
2026-03-14 14:09:19,051 INFO [hisepy.logging:175] logging 3852 133323142518592 Calling lookup_queryable_fields
2026-03-14 14:09:21,527 INFO [hisepy.logging:208] logging 3852 133323142518592 Finished lookup_queryable_fields, success=True, time_elapsed=1.031s
2026-03-14 14:09:21,529 INFO [hisepy.logging:175] logging 3852 133323142518592 Calling lookup_queryable_fields
2026-03-14 14:09:23,767 INFO [hisepy.logging:208] logging 3852 133323142518592 Finished lookup_queryable_fields, success=True, time_elapsed=0.990s
2026-03-14 14:09:34,941 INFO [hisepy.logging:208] logging 3852 133323142518592 Finished get_file_descriptors, success=True, time_elapsed=14.430s
2026-03-14 14:09:35,242 INFO [hisepy.logging:175] logging 3852 133323142518592 Calling get_file_descriptors
2026-03-14 14:09:35,243 INFO [hisepy.logging:175] logging 3852 133323142518592 Calling lookup_queryable_fields
2026-03-14 1

We only need some of the columns that are provided by HISE to find the most recent version and batch for each sample kit.

Note that the file time stamp is embedded in the file.name - see the helper function above for parsing.

In [ ]:
keep_cols = [
    'file.availability',
    'file.batchID',
    'file.id',
    'file.majorVersion',
    'file.name',
    'file.panel',
    'sample.sampleKitGuid',
]

The output of our chunk-based queries, above, is a list of data frames per chunk. We'll select columns and concatenate these chunks here.

`desc` is short for `descriptors`.

In [35]:
common_pt1_list = []
for df in pt1_list:
    common_pt1_list.append(df.select(keep_cols))

In [36]:
desc = pl.concat(common_pt1_list)
desc.shape

(1719, 7)

Now we'll do the filtering for each kit to get the latest batch and time stamp. We'll also keep track of any sample kits that are in the sample metadata but weren't found in our HISE file queries.

In [24]:
missing_kits = {}

In [25]:
kit_list = []
missing_list = []
for sample_kit in meta['sample.sampleKitGuid']:
    kit_files = desc.filter(pl.col('sample.sampleKitGuid') == sample_kit)
    if kit_files.shape[0] == 0:
        print(f'No files for {sample_kit}; Skipping.')
        missing_list.append(sample_kit)
        continue
    else:
        # Filter for latest batch
        latest_batch = get_latest_batch(kit_files)
        kit_files = kit_files.filter(pl.col('file.batchID') == latest_batch)
        # Filter for most recent file
        kit_files = kit_files.with_columns(
            pl.Series(
                name = 'file_time',
                values = [file_to_time(x) for x in kit_files['file.name']]
            ).str.to_datetime("%Y-%m-%dT%H:%M:%S")
        ).sort('file_time', descending = True).head(1)

        kit_list.append(kit_files)

No files for KT02480; Skipping.


The output for each kit is a list again, so we'll concatenate these for downstream use.

In [26]:
pt1_desc = pl.concat(kit_list)

Are there any missing kits?

In [27]:
len(missing_list)

1

In [28]:
missing_kits['PT1'] = meta.filter(pl.col('sample.sampleKitGuid').is_in(missing_list))

We should get the same number of rows in the descriptors data frame as in the original metadata file if all sample kits are accounted for.

In [29]:
meta.shape

(868, 14)

In [30]:
pt1_desc.shape

(867, 8)

## Cache and assemble files

Split the file descriptors to get file ids, and run queries in chunks of 50 files.

In [31]:
pt1_chunks = pt1_desc.iter_slices(n_rows = 50)

In [32]:
pt1_fail_list = []
pt1_files = []

for pt1_chunk in pt1_chunks:
    ids = pt1_chunk['file.id'].to_list()
    
    try:
        fn = cache_chunk_once(ids)
        pt1_files = pt1_files + fn
    except:
        pt1_fail_list.append(pt1_chunk)

2026-03-14 14:13:57,022 INFO [hisepy.logging:175] logging 3852 133323142518592 Calling cache_files
2026-03-14 14:14:21,713 INFO [hisepy.logging:208] logging 3852 133323142518592 Finished cache_files, success=True, time_elapsed=23.333s
2026-03-14 14:14:21,810 INFO [hisepy.logging:175] logging 3852 133323142518592 Calling cache_files
2026-03-14 14:16:19,392 INFO [hisepy.logging:208] logging 3852 133323142518592 Finished cache_files, success=True, time_elapsed=114.267s
2026-03-14 14:16:19,785 INFO [hisepy.logging:175] logging 3852 133323142518592 Calling cache_files
2026-03-14 14:16:48,372 INFO [hisepy.logging:208] logging 3852 133323142518592 Finished cache_files, success=True, time_elapsed=27.212s


In the end, we should get the same number of files as we had in pt1_desc:

In [33]:
pt1_desc.shape

(867, 8)

In [37]:
len(pt1_files)

867

### Transfer and structure outputs

We'll make a hierarchical file structure that will enable all of the panels and file groups to be untarred into the same folder structure.

File bundles will contain all samples for each age + sex + cmv group, and will have a metadata file for each group. We also need to have a file defining the panel in each bundle so that it's included in any particular bundle.

An example for one panel is shown below:

```
sound-life_flow-cytometry/
  PT1_panel/
    older-adult_female_cmv-negative/
      <subject>_<sample_kit>_<visit_name>_unmixed.fcs
    older-adult_female_cmv-positive/
    older-adult_male_cmv-negative/
    older-adult_male_cmv_positive/
    young-adult_female_cmv-negative/
    young-adult_female_cmv-positive/
    young-adult_male_cmv-negative/
    young-adult_male_cmv_positive/
```

First, we'll add the sample metadata to our descriptors so we can split files into the groups above.

In [38]:
file_df = pt1_desc.join(
    meta,
    how = 'left',
    on = 'sample.sampleKitGuid'
)

In [39]:
file_df.shape

(867, 21)

### Move files into a stage for .tar

Here, we'll move the files from the cache into the path structure defined above.

In [40]:
in_root = '/home/workspace/input/1918706177/'
out_base = 'sound-life_flow-cytometry/PT1_panel/'

Associate each uuid with the file location in the cache

In [41]:
all_cached_files = {}
for project in os.listdir(in_root):
    for uuid in pt1_desc['file.id']:
        if os.path.isdir(f'{in_root}/{project}/{uuid}'):
            all_cached_files[uuid] = find_nested_file(f'{in_root}/{project}/{uuid}')

Add cache file names to the file descriptors

In [42]:
panel_files = []

for uuid in pt1_desc['file.id']:
    if uuid in all_cached_files.keys():
        panel_files.append(all_cached_files[uuid])
    else:
        panel_files.append(None)

file_df = file_df.with_columns(
    pl.Series(
        name = 'in_file',
        values = panel_files
    )
)

Build directory structure file names. We'll make 3 columns here:

`out_dir`: The subdirectory based on:  
- `subject.ageGroup`\_`subject.biologicalSex`_`subject.cmv`/

`out_file`: The full target file name, based on:  
- `out_dir`/`subject.subjectGuid`\_`sample.sampleKitGuid`_`sample.visitName`_unmixed.fcs

`file.name`: The file path relative to the location of the metadata files in the file structure. This is the file name we'll provide to users.

In [43]:
file_df = file_df.with_columns(
    pl.col('subject.ageGroup').str.replace(' ', '-'),
    pl.col('sample.visitName').str.replace_all(' ', '-')
).with_columns(
    pl.concat_str(
        pl.lit(out_base),
        pl.col('subject.ageGroup').str.to_lowercase(),
        pl.lit('_'),
        pl.col('subject.biologicalSex').str.to_lowercase(),
        pl.lit('_cmv-'),
        pl.col('subject.cmv').str.to_lowercase(),
        pl.lit('/')
    ).alias('out_dir')
).with_columns(
    pl.concat_str(
        pl.col('out_dir'),
        pl.col('subject.subjectGuid'),
        pl.lit('_'),
        pl.col('sample.sampleKitGuid'),
        pl.lit('_'),
        pl.col('sample.visitName').str.to_lowercase(),
        pl.lit('_unmixed.fcs')
    ).alias('out_file')
).with_columns(
    pl.col('out_file').str.replace('sound-life_flow-cytometry/', '').alias('file.name')
)

### Make staging directories and move files

Make the output subdirectories

In [44]:
for out_dir in file_df['out_dir'].unique().to_list():
    if out_dir is not None:
        if not os.path.isdir(out_dir):
            os.makedirs(out_dir)

Copy the files to the staging directories

In [45]:
for in_file, out_file in zip(file_df['in_file'], file_df['out_file']):
    if not out_file is None:
        if not os.path.isfile(out_file):
            shutil.copy(in_file, out_file)

Add the panel metadata file

In [46]:
fcs_panel_file = 'sound-life_flow-cytometry/PT1_panel_feature_metadata.csv'
shutil.copy(panel_csv, fcs_panel_file)

'sound-life_flow-cytometry/PT1_panel_feature_metadata.csv'

## Get file-specific stats from within the .fcs files

In [47]:
file_df = file_df.with_columns(
    pl.Series(
        name = 'flow.event_count',
        values = [flowio.FlowData(f).event_count for f in file_df['out_file']]
    )
)

We'll rename the columns specific to the flow run to prepend `flow.` instead of `file.`

In [48]:
file_df = file_df.rename({
    'file.panel': 'flow.panel',
    'file.batchID': 'flow.batchID'
})

Select the file and sample metadata columns we want to retain for download.

We'll also revert the space to dash replacement we used to generate filenames.

In [49]:
file_df = file_df.select(
    'out_dir', 'file.name', 'file.id', 
    'flow.panel', 'flow.batchID', 'flow.event_count',
    'subject.ageGroup', 'subject.biologicalSex', 'subject.cmv', 
    'cohort.cohortGuid', 
    'subject.subjectGuid', 'subject.birthYear', 'subject.ageAtFirstDraw', 'subject.race', 'subject.ethnicity',
    'sample.sampleKitGuid', 'sample.visitName', 'sample.drawYear', 'sample.subjectAgeAtDraw', 'sample.daysSinceFirstVisit'
).with_columns(
    pl.col('subject.ageGroup').str.replace('-', ' ')
)

In [50]:
file_df.head()

out_dir,file.name,file.id,flow.panel,flow.batchID,flow.event_count,subject.ageGroup,subject.biologicalSex,subject.cmv,cohort.cohortGuid,subject.subjectGuid,subject.birthYear,subject.ageAtFirstDraw,subject.race,subject.ethnicity,sample.sampleKitGuid,sample.visitName,sample.drawYear,sample.subjectAgeAtDraw,sample.daysSinceFirstVisit
str,str,str,str,str,i64,str,str,str,str,str,i64,i64,str,str,str,str,str,i64,i64
"""sound-life_flow-cytometry/PT1_…","""PT1_panel/young-adult_female_c…","""64a50723-398b-464b-b311-785440…","""PT1""","""B151""",294264,"""Young Adult""","""Female""","""Negative""","""BR1""","""BR1001""",1987,32,"""Caucasian""","""Non-Hispanic origin""","""KT00001""","""Flu-Year-1-Day-0""","""2019""",32,0
"""sound-life_flow-cytometry/PT1_…","""PT1_panel/young-adult_male_cmv…","""0b099c21-8033-4520-9563-0153b0…","""PT1""","""B151""",479416,"""Young Adult""","""Male""","""Negative""","""BR1""","""BR1002""",1991,28,"""Caucasian""","""Non-Hispanic origin""","""KT00002""","""Flu-Year-1-Day-0""","""2019""",28,0
"""sound-life_flow-cytometry/PT1_…","""PT1_panel/young-adult_female_c…","""adf1e091-983f-4d51-a113-f9e046…","""PT1""","""B151""",312216,"""Young Adult""","""Female""","""Negative""","""BR1""","""BR1003""",1989,30,"""Caucasian""","""Non-Hispanic origin""","""KT00003""","""Flu-Year-1-Day-0""","""2019""",30,0
"""sound-life_flow-cytometry/PT1_…","""PT1_panel/young-adult_male_cmv…","""c6986a7f-6c47-4783-990b-c34388…","""PT1""","""B151""",325872,"""Young Adult""","""Male""","""Negative""","""BR1""","""BR1004""",1989,30,"""Caucasian""","""Non-Hispanic origin""","""KT00004""","""Flu-Year-1-Day-0""","""2019""",30,0
"""sound-life_flow-cytometry/PT1_…","""PT1_panel/young-adult_female_c…","""f15fa9ba-e5d9-42b7-85bb-83a3e0…","""PT1""","""B151""",375936,"""Young Adult""","""Female""","""Negative""","""BR1""","""BR1005""",1992,27,"""Caucasian""","""Non-Hispanic origin""","""KT00006""","""Flu-Year-1-Day-0""","""2019""",27,0


In [51]:
file_df['out_dir'][0]

'sound-life_flow-cytometry/PT1_panel/young-adult_female_cmv-negative/'

Summarize subjects, samples, and events per group

In [52]:
file_df.group_by(
    ['subject.ageGroup', 'subject.biologicalSex', 'subject.cmv']
).agg(
    pl.col('subject.subjectGuid').unique().len().alias('n_subjects'),
    pl.col('sample.sampleKitGuid').len().alias('n_samples'),
    pl.col('flow.batchID').unique().len().alias('n_batches'),
    pl.col('flow.event_count').sum().alias('n_events'),
).sort(['subject.ageGroup', 'subject.biologicalSex', 'subject.cmv'])

subject.ageGroup,subject.biologicalSex,subject.cmv,n_subjects,n_samples,n_batches,n_events
str,str,str,u32,u32,u32,i64
"""Older Adult""","""Female""","""Negative""",10,91,45,30046003
"""Older Adult""","""Female""","""Positive""",17,163,66,54364732
"""Older Adult""","""Male""","""Negative""",12,116,52,36919240
"""Older Adult""","""Male""","""Positive""",8,80,40,27284008
"""Young Adult""","""Female""","""Negative""",18,161,50,59015871
"""Young Adult""","""Female""","""Positive""",10,76,35,29069298
"""Young Adult""","""Male""","""Negative""",12,107,45,36806056
"""Young Adult""","""Male""","""Positive""",9,73,29,27603561


In [53]:
file_df['flow.batchID'].unique().len()

102

### Build final output files and .tar bundles

Save the full set of metadata to both the archive staging folder, and as a separate file to store in HISE.

We'll drop the `out_dir` column before saving - this won't be very useful to end users, but we'll need it to write the group-specific metadata files. 

`file.name` will be relative to the final metadata file path.

In [54]:
fcs_meta_file = 'sound-life_flow-cytometry/PT1_fcs_sample_metadata_all.csv'
file_df.drop('out_dir').write_csv(fcs_meta_file)

out_fcs_meta = 'output/PT1_fcs_sample_metadata_all_{d}.csv'.format(d = date.today())
file_df.drop('out_dir').write_csv(out_fcs_meta)

In [55]:
out_tarfiles = []
for out_dir in file_df['out_dir'].unique():
    if out_dir is not None:
        out_base = os.path.basename(re.sub('/$','',out_dir))

        out_meta_file = f'sound-life_flow-cytometry/PT1_fcs_sample_metadata_{out_base}.csv'
        out_meta = file_df.filter(
            pl.col('out_dir') == out_dir
        ).drop('out_dir')
        
        out_meta.write_csv(out_meta_file)

        d = date.today()
        out_tarfile = f'output/sound-life_flow-cytometry_PT1_{out_base}_{d}.tar'
        out_tarfiles.append(out_tarfile)

        with tarfile.open(out_tarfile, 'w') as tar:
            tar.add(fcs_panel_file)
            tar.add(fcs_meta_file)
            tar.add(out_meta_file)
            for fcs_file in out_meta['file.name']:
                tar.add(f'sound-life_flow-cytometry/{fcs_file}')

## Upload .fcs data to HISE

Finally, we'll use `hisepy.upload.upload_files()` to send a copy of our output to HISE to use for downstream analysis steps.

In [56]:
study_space_uuid = 'de025812-5e73-4b3c-9c3b-6d0eac412f2a'
title = 'Sound Life PT1 panel .fcs files {d}'.format(d = date.today())

In [57]:
search_id = element_id()
search_id

'francium-sodium-cesium'

In [58]:
in_files = [meta_uuid, panel_uuid] + pt1_desc['file.id'].to_list()
len(in_files)

869

In [59]:
out_files = [out_fcs_meta] + out_tarfiles
out_files

['output/PT1_fcs_sample_metadata_all_2026-03-14.csv',
 'output/sound-life_flow-cytometry_PT1_older-adult_female_cmv-negative_2026-03-14.tar',
 'output/sound-life_flow-cytometry_PT1_young-adult_male_cmv-positive_2026-03-14.tar',
 'output/sound-life_flow-cytometry_PT1_older-adult_female_cmv-positive_2026-03-14.tar',
 'output/sound-life_flow-cytometry_PT1_young-adult_male_cmv-negative_2026-03-14.tar',
 'output/sound-life_flow-cytometry_PT1_older-adult_male_cmv-negative_2026-03-14.tar',
 'output/sound-life_flow-cytometry_PT1_young-adult_female_cmv-positive_2026-03-14.tar',
 'output/sound-life_flow-cytometry_PT1_older-adult_male_cmv-positive_2026-03-14.tar',
 'output/sound-life_flow-cytometry_PT1_young-adult_female_cmv-negative_2026-03-14.tar']

In [60]:
import session_info
session_info.show()

In [61]:
hisepy.upload.upload_files(
    files = out_files,
    study_space_id = study_space_uuid,
    title = title,
    input_file_ids = in_files,
    destination = search_id
)

2026-03-14 15:23:31,675 INFO [hisepy.logging:175] logging 3852 133323142518592 Calling upload_files


Please provide input of comma separated sample ids for the files being uploaded. If you do not have any sample ids, press enter:  


2026-03-14 15:24:36,290 INFO [hisepy.logging:175] logging 3852 133323142518592 Calling get_default_store
2026-03-14 15:24:38,744 INFO [hisepy.logging:208] logging 3852 133323142518592 Finished get_default_store, success=True, time_elapsed=0.631s
2026-03-14 15:25:33,317 INFO [hisepy.logging:175] logging 3852 133323142518592 Calling conda_env_builds
2026-03-14 15:25:33,318 INFO [hisepy.logging:53] utils 3852 133323142518592 Starting conda environment build validation...
2026-03-14 15:25:33,705 INFO [hisepy.logging:75] utils 3852 133323142518592 Exporting conda environment from /home/workspace/environment/minimalv2...
2026-03-14 15:25:37,089 INFO [hisepy.logging:88] utils 3852 133323142518592 Removing hisepy references from exported environment file...
2026-03-14 15:25:37,097 INFO [hisepy.logging:99] utils 3852 133323142518592 Creating temporary conda environment at /tmp/conda_env_test_e3ugrdh2/env_b48f7f5d962b487d9d72895498c02614...
2026-03-14 15:26:51,454 INFO [hisepy.logging:119] utils

{'Message': 'General Okay-ness',
 'VisualizationId': '00000000-0000-0000-0000-000000000000',
 'AbstractionId': '00000000-0000-0000-0000-000000000000',
 'TraceId': '453bab68-36dd-4a00-a7e1-01ec9df3e8e4',
 'ProcessId': 'c370606c-eebb-4d8b-8dc4-d35db0307f0b',
 'WorkflowId': '83fea3c4-c30a-4436-bdcd-55f95561d5c2',
 'FileIds': ['d7dd1eb3-915b-474f-8ecc-4cb3abf83542',
  'cedee26e-e855-4670-8096-4a35bdc971a4',
  '80318217-6df9-42d1-a782-ef9e9e937255',
  '86e988b6-d3dc-4ecd-9046-af2e85861d3c',
  'e7f7a11b-3f49-412d-a6af-fd965a61605e',
  'a8aa552d-d6ec-47e4-a53c-2c1c32800fba',
  '68b6cd1b-f94d-49e7-b02e-d2beedd8869a',
  'b7a46f24-1a33-4600-b2cc-080e6935f661',
  '16cf6f1a-fbcd-48c0-9790-9ae9feea87bd']}